# Wyzwanie: Analiza tekstu o Data Science

W tym przykładzie wykonajmy proste ćwiczenie obejmujące wszystkie kroki tradycyjnego procesu data science. Nie musisz pisać kodu, możesz po prostu kliknąć w poniższe komórki, aby je wykonać i obserwować wynik. Jako wyzwanie zachęcamy do wypróbowania tego kodu na różnych danych.

## Cel

W tej lekcji omawialiśmy różne pojęcia związane z Data Science. Spróbujmy odkryć więcej powiązanych pojęć, wykonując **text mining**. Zaczniemy od tekstu o Data Science, wydobędziemy z niego słowa kluczowe, a następnie spróbujemy zwizualizować wynik.

Jako tekst wykorzystam stronę o Data Science z Wikipedii:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Krok 1: Pobieranie danych

Pierwszym krokiem w każdym procesie analizy danych jest pobranie danych. Użyjemy do tego biblioteki `requests`:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Krok 2: Przekształcanie danych

Następnym krokiem jest konwersja danych na formę odpowiednią do przetwarzania. W naszym przypadku pobraliśmy kod źródłowy HTML ze strony i musimy go przekształcić na zwykły tekst.

Istnieje wiele sposobów, aby to zrobić. Użyjemy [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), popularnej biblioteki Pythona do parsowania HTML. BeautifulSoup pozwala nam celować w konkretne elementy HTML, dzięki czemu możemy skupić się na głównej treści artykułu z Wikipedii i ograniczyć niektóre menu nawigacyjne, panele boczne, stopki i inne nieistotne treści (choć niektóry tekst szablonowy może nadal pozostać).


Najpierw musimy zainstalować bibliotekę BeautifulSoup do parsowania HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Krok 3: Uzyskiwanie wglądów

Najważniejszym krokiem jest przekształcenie naszych danych w formę, z której możemy wyciągnąć wglądy. W naszym przypadku chcemy wydobyć słowa kluczowe z tekstu i zobaczyć, które słowa kluczowe są bardziej znaczące.

Użyjemy biblioteki Pythona o nazwie [RAKE](https://github.com/aneesha/RAKE) do ekstrakcji słów kluczowych. Najpierw zainstalujmy tę bibliotekę, jeśli nie jest jeszcze obecna: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Główna funkcjonalność jest dostępna z obiektu `Rake`, który możemy dostosować za pomocą kilku parametrów. W naszym przypadku ustawimy minimalną długość słowa kluczowego na 5 znaków, minimalną częstość występowania słowa kluczowego w dokumencie na 3, oraz maksymalną liczbę słów w słowie kluczowym na 2. Śmiało eksperymentuj z innymi wartościami i obserwuj rezultat.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Otrzymaliśmy listę terminów wraz z przypisanym stopniem ważności. Jak widać, najistotniejsze dziedziny, takie jak uczenie maszynowe i big data, zajmują na liście najwyższe pozycje.

## Krok 4: Wizualizacja wyniku

Ludzie najlepiej interpretują dane w formie wizualnej. Dlatego często ma sens zwizualizowanie danych, aby wyciągnąć wnioski. Możemy użyć biblioteki `matplotlib` w Pythonie, aby narysować prosty rozkład słów kluczowych wraz z ich istotnością:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Istnieje jednak jeszcze lepszy sposób na wizualizację częstotliwości słów - za pomocą **Chmury słów**. Będziemy musieli zainstalować kolejną bibliotekę, aby narysować chmurę słów z naszej listy słów kluczowych.


In [ ]:
!{sys.executable} -m pip install wordcloud

Obiekt `WordCloud` odpowiada za przyjęcie oryginalnego tekstu lub wstępnie obliczonej listy słów wraz z ich częstotliwościami i zwraca obraz, który następnie można wyświetlić za pomocą `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Możemy też przekazać oryginalny tekst do `WordCloud` - zobaczmy, czy uda nam się uzyskać podobny wynik:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Jak widać, chmura słów wygląda teraz bardziej imponująco, ale zawiera także wiele szumu (np. niepowiązane słowa takie jak `Retrieved on`). Ponadto otrzymujemy mniej fraz kluczowych złożonych z dwóch słów, takich jak *data scientist* czy *computer science*. Wynika to z faktu, że algorytm RAKE radzi sobie znacznie lepiej z wyborem dobrych fraz kluczowych z tekstu. Ten przykład pokazuje, jak ważne jest przetwarzanie i czyszczenie danych, ponieważ przejrzysty obraz na końcu pozwoli nam podejmować lepsze decyzje.

W tym ćwiczeniu przeszliśmy przez prosty proces wydobywania znaczenia z tekstu Wikipedii w postaci fraz kluczowych i chmury słów. Ten przykład jest dość prosty, ale dobrze ilustruje wszystkie typowe kroki, jakie wykonuje data scientist podczas pracy z danymi, począwszy od pozyskania danych, aż do wizualizacji.

W naszym kursie omówimy wszystkie te kroki szczegółowo. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
